# Project: Breast Cancer (SEER) & FUTURE-AI

This notebook prepares the data **identically for all groups**, so results stay comparable even if you choose different FUTURE-AI criteria. It contains:

1. Dataset import
2. Label creation (5-year Overall Survival)
3. Train/test split and Cross-Validation
4. Data pre-processing

**From here on, the project is yours.** You will need to add:
- **EDA** — explore the dataset: distributions, subgroups, any data quality issues, relationships between variables and with the outcome
- **Machine Learning pipeline** — feature engineering/selection, model choice and comparison, hyperparameter tuning (using the Cross-Validation already set up), final evaluation on the test set
- **FUTURE-AI assessment** — Fairness, Universality, Traceability, Usability, Robustness, Explainability and design a concrete analysis for each

**Important rule**: the test set created in this notebook must be used **only for the final evaluation** of the chosen model. To compare models/hyperparameters during development, use Cross-Validation on the train set — never the test set, otherwise the final results will be optimistically biased ("procedural" data leakage).


## 1. Dataset import

In [ ]:
import pandas as pd
import numpy as np

SEED = 42
np.random.seed(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

### Caricamento del Dataset (Google Drive / Locale)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DATASET_DIR = '/content/drive/MyDrive/progettoneuro'

In [ ]:
import pandas as pd
import os

dataset_path = "/content/drive/MyDrive/progettoneuro/Copia di SEER Breast Cancer Dataset .csv"

df = pd.read_csv(dataset_path)

print("Raw shape:", df.shape)
df.head()
#the dataset has 4000 patient, we want to predict if they will be alive in 5 years


The file contains two small technical artifacts that must be removed before any analysis (these are not part of the EDA you will do — they are defects of the CSV file, not of the clinical content):
- an unnamed, completely empty column;
- 1 duplicate row.

In [ ]:
df = df.drop(columns=["Unnamed: 3"]).copy() #to remove empty colums
df.columns = [c.strip() for c in df.columns]  # removes accidental whitespace in column names
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after minimal cleaning:", df.shape)
assert df.isna().sum().sum() == 0, "there should be no missing values left" #there are no any missing value
df.head()

### Data dictionary

| Column | Meaning |
|---|---|
| `Age` | Age at diagnosis (years) |
| `Race` | Ethnicity (SEER categories: White / Black / Other) |
| `Marital Status` | Marital status at diagnosis |
| `T Stage` | Extent of the primary tumor (AJCC), T1 (small) → T4 (extensive) |
| `N Stage` | Regional lymph node involvement, N1 → N3 (increasing) |
| `6th Stage` | Overall AJCC 6th edition stage (combines T, N, and metastatic status) |
| `Grade` | Histologic differentiation grade of the tumor (I = well differentiated → IV = anaplastic) |
| `A Stage` | Regional (localized/loco-regional) vs Distant (metastatic) |
| `Tumor Size` | Tumor diameter (mm) |
| `Estrogen Status` / `Progesterone Status` | Hormone receptor status (Positive/Negative) |
| `Regional Node Examined` | No. of lymph nodes surgically examined |
| `Reginol Node Positive` | No. of lymph nodes found positive (with metastasis) |
| `Survival Months` | Months of observed follow-up |
| `Status` | Status at the end of follow-up: Alive (alive/censored) or Dead (deceased) |

`Survival Months` and `Status` are **only** used to build the label — they are not legitimate model features (using them would be data leakage: they would use the very outcome you're trying to predict).

## 2. Label creation — 5-year Overall Survival

The task is: **predict whether a patient will have died within 5 years (60 months) of diagnosis**, using only clinico-pathological information available at diagnosis/initial staging.

Not all patients can be used: anyone still "Alive" with a shorter follow-up than 60 months is still under observation — their 5-year outcome is not known (censoring), and they must be excluded rather than forced into one of the two classes.

- **label = 1** (death within 5 years): `Status == "Dead"` and `Survival Months < 60`
- **label = 0** (survives to 5 years): `Survival Months >= 60` (regardless of what happens afterwards)
- **excluded**: `Status == "Alive"` and `Survival Months < 60` (unknown outcome)


In [ ]:
HORIZON = 60  # months

survived = df[df["Survival Months"] >= HORIZON].copy()
survived["label"] = 0

died_early = df[(df["Status"] == "Dead") & (df["Survival Months"] < HORIZON)].copy()
died_early["label"] = 1

excluded = df[(df["Status"] == "Alive") & (df["Survival Months"] < HORIZON)]

task_df = pd.concat([survived, died_early], ignore_index=True)

print(f"Evaluable patients: {len(task_df)}  |  excluded (censored before 5 years): {len(excluded)} "
      f"({len(excluded)/len(df)*100:.1f}% of the original cohort)")
print(task_df["label"].value_counts(normalize=True).rename("share").round(3))

## 3. Train/test split and Cross-Validation

Stratified split (keeps the same proportion of events in train and test) with a fixed seed, so all groups start from the same data.

The columns `Survival Months`, `Status` (used to build the label) and the `label` itself are separated out from `X`. **`6th Stage` is not removed here**: it's one of the questions you should ask yourselves during EDA and feature engineering (hint: check its relationship with `T Stage`/`N Stage`/`A Stage`).

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

LEAKAGE_COLS = ["Survival Months", "Status", "label"]  # do NOT use as a feature

X = task_df.drop(columns=LEAKAGE_COLS)
y = task_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Event prevalence (label=1) — train:", round(y_train.mean(), 3), " test:", round(y_test.mean(), 3))

In [ ]:
# Cross-Validation to use for comparing models/hyperparameters ON THE TRAIN SET
# (never on the test set: that is reserved for final evaluation)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

## 4. Data pre-processing

We define a common preprocessor (scaling of numeric variables + one-hot encoding of categoricals), to be used as the first step of any scikit-learn `Pipeline` you build. It must **always be fit on the train set only** (inside CV or on the final train set) and never on the whole dataset, to avoid data leakage between train and test.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

NUM_COLS = ["Age", "Tumor Size", "Regional Node Examined", "Reginol Node Positive"]
CAT_COLS = ["Race", "Marital Status", "T Stage", "N Stage", "6th Stage",
            "Grade", "A Stage", "Estrogen Status", "Progesterone Status"]
assert set(NUM_COLS) | set(CAT_COLS) == set(X.columns)

def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ])

In [ ]:
# sanity check: the preprocessor works and produces the expected output
preprocessor = make_preprocessor()
X_train_transformed = preprocessor.fit_transform(X_train)
print("X_train shape after preprocessing:", X_train_transformed.shape)
print("Example of generated feature names:", list(preprocessor.get_feature_names_out()[:5]), "...")

**Note**: the encoder here uses `OneHotEncoder(handle_unknown="ignore")` without dropping any reference level — this is a neutral default choice. If your pipeline uses a linear model and you want easily interpretable coefficients, consider whether and how to handle reference categories (e.g. `drop="first"`) — this is one of the design decisions you are expected to justify.

## From here on, it's up to you

You have available: `df` (cleaned dataset), `task_df` (with the label), `X_train`, `X_test`, `y_train`, `y_test`, the `cv` object, and `make_preprocessor()`.

Project checklist:
- [ ] **EDA**: explore `df`/`task_df` — distributions, subgroups, any dataset issues or limitations, relationships between clinical variables and the outcome
- [ ] **Feature engineering / selection**: which columns to keep? is there redundancy? do outliers or transformations need handling?
- [ ] **ML pipeline**: build one or more `Pipeline`s (`preprocessor` → optional selection → model), compare models/hyperparameters with `cv` on the train set, then evaluate **once** on the test set
- [ ] **Metrics**: justify your choice of evaluation metrics (watch out for class imbalance — accuracy alone is misleading)
- [ ] **FUTURE-AI**: choose 2 criteria among Fairness, Universality, Traceability, Usability, Robustness, Explainability, and design a quantitative and/or qualitative analysis for each, justifying your choice
